# InferOps Week 3: Quantization (speed, memory and quality)

**Question:** if each weight takes fewer bytes, does the model get faster and cheaper, and what does it cost in answer quality?

Two tracks on the same model (Qwen2.5-3B-Instruct):

| Track | Engine | Precisions |
|---|---|---|
| A | our HF engine + bitsandbytes | FP16, INT8, NF4 (4-bit) |
| B | vLLM | FP16, AWQ (4-bit), GPTQ-INT8 (Qwen's official quantized checkpoints) |

Every configuration gets the same **speed** test (medium prompts) and the same **quality** test (GSM8K math).

**Predict before you run:** decode reads every weight once per token, so 4-bit weights mean 4x fewer bytes per token.
Should 4-bit be ~4x faster? Write your guess down; the answer differs between track A and B, and *why* is the lesson.

**Setup:** Kaggle, GPU T4 x2, Internet on. **Budget:** about 80-100 GPU minutes. Stop the session when done.

## 1. Get the code

In [ ]:
import os
%cd /kaggle/working
if os.path.exists("inferops"):
    !git -C inferops pull
else:
    !git clone https://github.com/saideepa05/inferops.git
%cd /kaggle/working/inferops
!pip install -q -e ".[hf,dev,quant]"
!python -m pytest -q 2>&1 | tail -1

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 2. Experiment configuration

Quality on the HF engine uses fewer questions because that engine answers one at a time
(about 10 s per question). Fewer questions means a wider confidence interval; the report shows it.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
OUT_DIR = "results/week3"
os.makedirs(OUT_DIR, exist_ok=True)

DATASET, MAX_TOKENS = "medium", 128      # ~350-token prompts, fixed 128-token outputs
SLO_TTFT_MS, SLO_TPOT_MS = 1000, 100

HF_CONCURRENCY, HF_REQUESTS = "1", 16      # serial engine: concurrency 1 shows its true speed
VLLM_CONCURRENCY, VLLM_REQUESTS = "1,8,32", 64

HF_QUALITY_N = 50                        # ~8-15 min per config on the serial engine
VLLM_QUALITY_N = 200                     # batched engine: ~1-2 min

## 3. Helpers

In [ ]:
import json, re, subprocess, time, urllib.request

def start_server(cmd, env_extra, port, log_path, timeout_s=900):
    """Launch a server in the background and wait until /health answers."""
    log = open(log_path, "w")
    proc = subprocess.Popen(cmd, env=os.environ | env_extra, stdout=log, stderr=subprocess.STDOUT)
    start = time.time()
    while time.time() - start < timeout_s:
        if proc.poll() is not None:
            raise RuntimeError(f"Server exited early. Last log lines:\n{open(log_path).read()[-3000:]}")
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=2)
            print(f"Ready after {time.time() - start:.0f}s on port {port}")
            return proc
        except Exception:
            time.sleep(3)
    proc.terminate()
    raise TimeoutError(f"Server not ready after {timeout_s}s; see {log_path}")

def stop_server(proc):
    proc.terminate()
    try:
        proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        proc.kill()
    time.sleep(5)  # let the driver release GPU memory before the next server

def gpu_mem_mib(index):
    out = subprocess.run(["nvidia-smi", f"--id={index}", "--query-gpu=memory.used",
                          "--format=csv,noheader,nounits"], capture_output=True, text=True)
    return float(out.stdout.strip())

def run_speed(label, port, concurrency, requests):
    !python -m bench.loadgen --url http://127.0.0.1:{port} --dataset {DATASET} --concurrency {concurrency} --num-requests {requests} --max-tokens {MAX_TOKENS} --slo-ttft-ms {SLO_TTFT_MS} --slo-tpot-ms {SLO_TPOT_MS} --label {label} --out {OUT_DIR}/{label}_speed.json

def run_quality(label, port, n, concurrency):
    !python -m bench.quality --url http://127.0.0.1:{port} --n {n} --concurrency {concurrency} --label {label} --out {OUT_DIR}/{label}_quality.json

def save_meta(label, **fields):
    json.dump(fields, open(f"{OUT_DIR}/{label}_meta.json", "w"), indent=2)
    print(label, fields)

# Part A: our HF engine + bitsandbytes

bitsandbytes stores weights in 8 or 4 bits and **dequantizes them back to FP16 inside every matrix multiply**.

**Predict:** memory for weights should be roughly 6.2 GB (FP16), 3.5 GB (INT8), 2.2 GB (NF4); embeddings and
some layers stay FP16, and the CUDA context adds ~0.4 GB. Speed: will NF4 be faster or slower than FP16?

Each config: start the server on GPU 0, record memory, run the speed test, run the quality test, stop.
A failing config prints its error and the loop continues.

In [ ]:
for quant in ["none", "int8", "nf4"]:
    label = f"hf-{'fp16' if quant == 'none' else quant}"
    print(f"\n===== {label} =====")
    try:
        proc = start_server(
            ["python", "-m", "uvicorn", "inferops.server:app", "--port", "8000", "--log-level", "warning"],
            {"CUDA_VISIBLE_DEVICES": "0", "INFEROPS_BACKEND": "hf", "INFEROPS_MODEL_ID": MODEL_ID,
             "INFEROPS_DEVICE": "cuda", "INFEROPS_DTYPE": "float16", "INFEROPS_QUANTIZATION": quant},
            port=8000, log_path=f"{label}.log")
    except Exception as exc:
        print(f"{label} failed to start: {exc}")
        continue
    try:
        save_meta(label, engine="hf", precision=label.removeprefix("hf-"), model=MODEL_ID,
                  loaded_mem_gb=gpu_mem_mib(0) / 1024)
        run_speed(label, 8000, HF_CONCURRENCY, HF_REQUESTS)
        run_quality(label, 8000, HF_QUALITY_N, concurrency=4)
    finally:
        stop_server(proc)

**Look before moving on:** compare TPOT across the three. If NF4 used a quarter of the memory but is *slower*,
that is not a bug. The GPU still does FP16 math, plus extra work to unpack 4-bit codes into FP16
for every layer on every token. bitsandbytes is built for **fitting** a model in memory, not for speed.

# Part B: vLLM

vLLM ships its own pinned PyTorch, so it goes into a **separate virtual environment** (Python 3.12,
because vLLM wheels may lag behind Kaggle's newest Python). Our benchmark tools keep running in the
notebook's environment and talk to vLLM over HTTP, which is exactly why the API is OpenAI-compatible.

The install takes 3-6 minutes.

In [ ]:
VLLM_ENV = "/kaggle/working/vllm-env"
!pip install -q uv
!uv venv -q {VLLM_ENV} --python 3.12
!uv pip install -q --python {VLLM_ENV}/bin/python vllm
!{VLLM_ENV}/bin/python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__)"

## Compatibility check: does vLLM run on a T4?

The T4 is an older GPU (compute capability 7.5: no bfloat16, no FlashAttention 2). Recent vLLM versions mostly target
newer GPUs, so this is the riskiest step of the week. Weeks 4-5 depend on its answer.

`VLLM_ARGS` and `VLLM_ENV_EXTRA` are used by every vLLM server below, so a fix found here applies everywhere.

In [ ]:
VLLM_ARGS = ["--dtype", "half", "--max-model-len", "4096", "--gpu-memory-utilization", "0.90"]
VLLM_ENV_EXTRA = {}

def start_vllm(model, label):
    return start_server([f"{VLLM_ENV}/bin/vllm", "serve", model, "--port", "8001", *VLLM_ARGS],
                        {"CUDA_VISIBLE_DEVICES": "1", **VLLM_ENV_EXTRA}, port=8001,
                        log_path=f"{label}.log", timeout_s=1200)

import httpx
proc, VLLM_OK = None, False
try:
    proc = start_vllm(MODEL_ID, "vllm-check")
    r = httpx.post("http://127.0.0.1:8001/v1/chat/completions", timeout=120, json={
        "model": MODEL_ID, "max_tokens": 60, "temperature": 0,
        "messages": [{"role": "user", "content": "Explain the KV cache in two sentences."}]}).json()
    print(r["choices"][0]["message"]["content"])
    print("vLLM WORKS on this T4. Skip the fallback cell.")
    VLLM_OK = True
except Exception as exc:
    print("vLLM FAILED:", str(exc)[-2000:])
    print("\nRun the fallback cell below, then re-run this cell.")
finally:
    if proc and proc.poll() is None:
        stop_server(proc)

### Fallback (only if the check failed)

Try these one at a time, re-running the check after each:

1. **Skip CUDA graph capture:** add `--enforce-eager` (slower, but rules out graph-capture problems on Turing).
2. **Pin an older vLLM** that still had the V0 engine with broader GPU support.

Note what failed and which fix worked: that's an engineering decision record for the docs.

In [ ]:
# Guarded: --enforce-eager turns off CUDA graphs and torch.compile, which makes vLLM much
# slower. Applying it when the check already passed would handicap every result below.
assert not VLLM_OK, "The check passed; do not apply the fallback."

# Option 1
VLLM_ARGS = VLLM_ARGS + ["--enforce-eager"]

# Option 2 (uncomment if option 1 is not enough)
# !uv pip install -q --python {VLLM_ENV}/bin/python "vllm==0.9.2"
# VLLM_ENV_EXTRA = {"VLLM_USE_V1": "0"}

## Run the three vLLM variants

vLLM reserves 90% of GPU memory up front, so `nvidia-smi` always looks full. The useful numbers are in its startup log:
**how much the weights take** and **how many tokens of KV cache fit in the rest**.

**Predict:** if 4-bit weights free ~4 GB, what happens to the KV cache size, and to how many requests can run at once?

In [ ]:
VARIANTS = {
    "vllm-fp16": MODEL_ID,
    "vllm-awq": "Qwen/Qwen2.5-3B-Instruct-AWQ",
    "vllm-gptq-int8": "Qwen/Qwen2.5-3B-Instruct-GPTQ-Int8",
}

def parse_vllm_log(path):
    text = open(path).read()
    grab = lambda pattern: (m := re.search(pattern, text)) and float(m.group(1).replace(",", ""))
    return {"weights_gb": grab(r"(?:weights take|loading took) ([\d.]+) ?GiB"),
            "kv_cache_tokens": grab(r"KV cache size: ([\d,]+) tokens"),
            "max_concurrency_at_4096": grab(r"Maximum concurrency for [\d,]+ tokens per request: ([\d.]+)")}

for label, model in VARIANTS.items():
    print(f"\n===== {label} =====")
    try:
        proc = start_vllm(model, label)
    except Exception as exc:
        print(f"{label} failed to start: {str(exc)[-1500:]}")
        continue
    try:
        save_meta(label, engine="vllm", precision=label.removeprefix("vllm-"), model=model,
                  enforce_eager="--enforce-eager" in VLLM_ARGS, **parse_vllm_log(f"{label}.log"))
        run_speed(label, 8001, VLLM_CONCURRENCY, VLLM_REQUESTS)
        run_quality(label, 8001, VLLM_QUALITY_N, concurrency=32)
    finally:
        stop_server(proc)

## 4. Results table

In [ ]:
import glob, pandas as pd

rows, EAGER = [], set()   # EAGER: configs run without CUDA graphs, flagged on the charts
for meta_path in sorted(glob.glob(f"{OUT_DIR}/*_meta.json")):
    label = os.path.basename(meta_path).removesuffix("_meta.json")
    meta = json.load(open(meta_path))
    if meta.get("enforce_eager"):
        EAGER.add(label)
    row = {"config": label, "engine": meta["engine"], "precision": meta["precision"],
           "weights_gb": meta.get("weights_gb") or meta.get("loaded_mem_gb"),
           "kv_cache_tokens": meta.get("kv_cache_tokens")}
    if os.path.exists(p := f"{OUT_DIR}/{label}_speed.json"):
        levels = {lv["concurrency"]: lv for lv in json.load(open(p))["levels"]}
        c1, top = levels[1], levels[max(levels)]
        row |= {"tpot_c1_ms": c1["tpot_s"]["p50"] * 1000, "tok_s_c1": c1["output_tok_s"],
                "max_conc": max(levels), "tok_s_max": top["output_tok_s"],
                "ttft_p95_max_ms": top["ttft_s"]["p95"] * 1000}
    if os.path.exists(p := f"{OUT_DIR}/{label}_quality.json"):
        q = json.load(open(p))["summary"]
        row |= {"gsm8k_acc": q["accuracy"] * 100, "ci_lo": q["ci95"][0] * 100, "ci_hi": q["ci95"][1] * 100,
                "n": q["n"], "truncated": q["truncated"]}
    rows.append(row)

order = ["hf-fp16", "hf-int8", "hf-nf4", "vllm-fp16", "vllm-gptq-int8", "vllm-awq"]
df = pd.DataFrame(rows).set_index("config").reindex([c for c in order if c in {r["config"] for r in rows}])
df.round(1)

**How to read it:**
- `weights_gb`: HF rows are total GPU memory after loading (includes ~0.4 GB CUDA context); vLLM rows are the weights alone.
- `tpot_c1_ms`: one user, pure decode speed. This is where "fewer bytes per weight" should show up.
- `tok_s_max`: throughput with many users. vLLM batches; our HF engine doesn't, so its "max" equals its c=1.
- `gsm8k_acc` with `ci_lo`-`ci_hi`: two configs only differ in quality if their intervals barely overlap.

## 5. Charts

In [ ]:
import matplotlib.pyplot as plt

COLORS = {"hf": "#2a78d6", "vllm": "#eb6834"}   # colour = engine, fixed in every panel
INK, MUTED, SURFACE = "#0b0b0b", "#52514e", "#fcfcfb"

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), facecolor=SURFACE)
panels = [("tok_s_c1", "Decode speed, 1 user (tokens/s)"),
          ("tok_s_max", "Throughput at max concurrency (tokens/s)"),
          ("gsm8k_acc", "GSM8K accuracy (%), 95% CI")]
x = range(len(df))
ticks = [f"{c} *" if c in EAGER else c for c in df.index]
for ax, (col, title) in zip(axes, panels):
    ax.set_facecolor(SURFACE)
    vals = df[col] if col in df else [0] * len(df)
    bars = ax.bar(x, vals, width=0.6, color=[COLORS[e] for e in df.engine],
                  edgecolor=SURFACE, linewidth=2)
    tops = vals
    if col == "gsm8k_acc" and "ci_lo" in df:
        ax.errorbar(x, df[col], yerr=[df[col] - df.ci_lo, df.ci_hi - df[col]],
                    fmt="none", ecolor=INK, elinewidth=1, capsize=4)
        tops = df.ci_hi   # label above the CI cap, not on it
    else:
        # Eager mode only affects speed, so hatch the speed bars it handicapped
        for b, c in zip(bars, df.index):
            if c in EAGER:
                b.set_hatch("///")
    for b, v, top in zip(bars, vals, tops):
        if pd.notna(v):
            ax.annotate(f"{v:.0f}", (b.get_x() + b.get_width() / 2, top), textcoords="offset points",
                        xytext=(0, 3), ha="center", fontsize=9, color=MUTED)
    ax.set_xticks(list(x), ticks, rotation=30, ha="right", color=MUTED)
    ax.set_title(title, color=INK, loc="left", fontsize=11)
    ax.set_ylim(bottom=0)
    ax.tick_params(colors=MUTED)
    ax.grid(True, axis="y", color="#e6e5e0", lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#c9c8c2")
fig.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in COLORS.values()],
           labels=["HF engine", "vLLM"], frameon=False, ncol=2, loc="upper right")
fig.suptitle(f"Quantization on a Tesla T4: {MODEL_ID}", color=INK, x=0.01, ha="left", fontsize=13)
fig.tight_layout()
if EAGER:
    fig.subplots_adjust(bottom=0.27)
    fig.text(0.01, 0.01, "* hatched: run with --enforce-eager (no CUDA graphs), so speed is understated. "
             "Accuracy is unaffected.", color=MUTED, fontsize=9)
fig.savefig(f"{OUT_DIR}/week3_charts.png", dpi=150, facecolor=SURFACE)
plt.show()

## 6. Write the report and package the results

In [ ]:
from datetime import datetime, timezone

def md_table(frame):
    cols = ["config", *frame.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for name, row in frame.iterrows():
        cells = [name] + ["" if pd.isna(v) else f"{v:.1f}" if isinstance(v, float) else str(v) for v in row]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)

vllm_version = subprocess.run([f"{VLLM_ENV}/bin/python", "-c", "import vllm; print(vllm.__version__)"],
                              capture_output=True, text=True).stdout.strip() or "not installed"
report = f"""# Week 3 report: quantization on a Tesla T4

Generated {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}

## Environment
| | |
|---|---|
| GPU | {torch.cuda.get_device_name(0)} |
| Model | {MODEL_ID} and Qwen's official AWQ / GPTQ-Int8 checkpoints |
| torch / transformers (HF track) | {torch.__version__} / {transformers.__version__} |
| vLLM | {vllm_version}, args: {" ".join(VLLM_ARGS)} {VLLM_ENV_EXTRA or ""} |

## Workload
- Speed: `{DATASET}` prompts, {MAX_TOKENS} output tokens (ignore_eos), temperature 0.
  HF concurrency {HF_CONCURRENCY} x {HF_REQUESTS} requests; vLLM concurrency {VLLM_CONCURRENCY} x {VLLM_REQUESTS} requests.
- Quality: first N GSM8K test questions, greedy decoding, up to 512 tokens.
  N = {HF_QUALITY_N} (HF) / {VLLM_QUALITY_N} (vLLM).

## Results
{md_table(df)}

![charts](week3_charts.png)

## vLLM on T4
_Did it run out of the box? What failed, what fixed it?_

## Findings
_Write these yourself. See the questions at the end of the notebook._

## Decision
_Which precision would you deploy on a T4, for which engine, and why?_
"""
open(f"{OUT_DIR}/report.md", "w").write(report)
!cp *.log {OUT_DIR}/ 2>/dev/null; true
!cd /kaggle/working/inferops && zip -qr /kaggle/working/week3_results.zip {OUT_DIR}
print(report)
print("Download week3_results.zip from the Output panel (right side, /kaggle/working).")

## 7. Questions to answer in your Findings

1. **HF track:** how did TPOT change from FP16 to INT8 to NF4? Explain the direction using what bitsandbytes does inside each matmul.
2. **vLLM track:** how did TPOT change from FP16 to AWQ? Compute the bandwidth ceiling for 4-bit weights
   (320 GB/s / (3.09B x 0.5 bytes)) and compare. Why can AWQ get faster when NF4 didn't?
3. **Memory:** how much did the weights shrink, and how much did vLLM's KV cache grow in return? What does that mean for concurrency?
4. **Quality:** did any precision lose accuracy *beyond the confidence interval*? How many questions would you need to detect a 2-point drop?
5. **Engines:** compare `hf-fp16` with `vllm-fp16` at concurrency 1 (same weights, same precision). Where does the difference come from?
6. **Throughput:** `tok_s_max` for vLLM vs HF. That gap is next week's topic (continuous batching).
7. **Decision:** which configuration would you ship for a chat product on T4s, and what would make you change your mind?